# Figure 1: Fit quality across read noise × peak QY — 2D surface

20,000 bootstrapped fits per grid point for ATTO 488, ATTO 565, and ATTO 647N.  
Each molecule emits 1,000 photons.  Molecule position is sampled uniformly within ±1 pixel of centre.  
Read noise is swept log-linearly from 0.01 to 10 RMS e⁻ (25 points).  
Peak pixel QY is swept linearly from 0.1 to 0.9 (9 points) by scaling the normalised Bayer efficiency curves.  
Outputs: 2D surfaces of fit yield, σ_xy, and σ_colour for each dye.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import types
import sys
from scipy.spatial.distance import cdist

sys.path.append('../..')
from src import IOFunctions
IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions
MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions
PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions
plotter = PlottingFunctions.Plotter(dark_background=False)

from src import ImageAnalysisFunctions
I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions
sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions
S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions
M_F = MaskFunctions.Mask_Functions()

from src.Multicolour_Simulation_Functions import FittingStrategy, SimulationConfig, CameraParameters

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20260401_184818.log
/tmp/ipykernel_2162698/4092055864.py:20: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter(dark_background=False)


In [2]:
# --- Camera calibration (median scalars from real Ximea calibration) ---
data_folder = '../../Camera_Calibrations/Ximea_Camera/'
gain     = IO.read_tiff(os.path.join(data_folder, 'gain.tif'))
offset   = IO.read_tiff(os.path.join(data_folder, 'offset.tif'))
rqe      = IO.read_tiff(os.path.join(data_folder, 'rqe.tif'))

gain_median   = float(np.median(gain))
offset_median = float(np.median(offset))
rqe_median    = float(np.median(rqe))

# --- Grid size ---
image_size = 12   # pixels (12×12 Bayer grid)
pixel_size = 69   # nm

# --- Bayer pixel QYs ---
R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])   # shape (3, n_wavelengths)

# Normalise to peak=1 so each peak_qy value scales the absolute efficiency
base_pixel_QYs = pixel_QYs / np.max(pixel_QYs)

masks = M_F.get_masks(size_x=image_size, size_y=image_size)

# --- Optical filters ---
notch_filter    = 'semrock-nf03-405-488-561-635e'
dichroic_mirror = 'semrock-di03-r405-488-561-635-t1-25x36'
filters = [dichroic_mirror, notch_filter]

# --- Smoothing function ---
smoothing_function = types.SimpleNamespace()
smoothing_function.args               = {'sigma': 1.5}
smoothing_function.extent             = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg           = 'image'

print(f'gain={gain_median:.3f}, offset={offset_median:.3f}, rqe={rqe_median:.4f}')
print(f'base_pixel_QYs peak: {np.max(base_pixel_QYs):.4f} (should be 1.0)')

gain=0.414, offset=32.346, rqe=1.0000
base_pixel_QYs peak: 1.0000 (should be 1.0)


In [3]:
# --- Simulation parameters ---
dyes           = ['ATTO 488', 'ATTO 565', 'ATTO 647N']
n_photons      = 1000
n_bootstrap    = 20000
n_photon_space = np.array([n_photons], dtype=float)

# Read-noise sweep: 25 log-spaced points from 0.01 to 10 RMS e-
read_noise_space = np.logspace(np.log10(0.01), np.log10(10.0), 50)

# Peak QY sweep: 9 linear points from 0.1 to 0.9
peak_qy_space = np.linspace(0.1, 0.9, 60)

save_folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/Figure1_MaxReadNoise_2D'
os.makedirs(save_folder, exist_ok=True)

print(f'Saving to:       {save_folder}')
print(f'Dyes:            {dyes}')
print(f'Read-noise range: {read_noise_space[0]:.4f} – {read_noise_space[-1]:.2f} RMS e-  ({len(read_noise_space)} points)')
print(f'Peak QY range:    {peak_qy_space[0]:.1f} – {peak_qy_space[-1]:.1f}  ({len(peak_qy_space)} points)')
print(f'Total grid points per dye: {len(read_noise_space) * len(peak_qy_space)}  ×  {n_bootstrap} bootstraps')

Saving to:       /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/Figure1_MaxReadNoise_2D
Dyes:            ['ATTO 488', 'ATTO 565', 'ATTO 647N']
Read-noise range: 0.0100 – 10.00 RMS e-  (50 points)
Peak QY range:    0.1 – 0.9  (60 points)
Total grid points per dye: 3000  ×  20000 bootstraps


In [ ]:
# --- Run bootstrap across all (dye, peak_qy, read_noise) grid points ---
n_total = len(dyes) * len(peak_qy_space) * len(read_noise_space)
count   = 0

for dye in dyes:
    dyestr = dye.replace(' ', '-').replace('/', '-')
    for peak_qy in peak_qy_space:
        scaled_pixel_QYs = base_pixel_QYs * peak_qy

        for rn in read_noise_space:
            count += 1
            flag = f'{dyestr}_qy_{peak_qy:.3f}_readnoise_{rn:.6f}_'

            camera_parameters_dict = {
                'gain':                np.full((image_size, image_size), gain_median),
                'offset':              np.full((image_size, image_size), offset_median),
                'variance':            np.full((image_size, image_size), rn ** 2),
                'readnoise':           np.full((image_size, image_size), rn),
                'rqe':                 np.full((image_size, image_size), rqe_median),
                'masks':               masks,
                'pixel_QYs':           scaled_pixel_QYs,
                'pixel_order':         ['B', 'G', 'R'],
                'pixel_order_indices': {'B': 0, 'G': 1, 'R': 2},
            }

            config = SimulationConfig(
                n_bootstrap=n_bootstrap,
                background_photons=5.0,
                background_colour=[1, 1, 1],
                NA=1.49,
                pixel_size=pixel_size,
                cpu_fraction=0.9,
                save_raw_results=True,
                subtractx0y0=False,
                saverawimages=False,
                use_stochastic_photons=True,
                verbose=False,
            )

            MSF.test_simulation_method(
                dye=dye,
                filters=filters,
                wavelength=wavelength,
                camera_parameters=camera_parameters_dict,
                save_folder=save_folder,
                n_photon_space=n_photon_space,
                smoothing_function=smoothing_function,
                strategy=FittingStrategy.STANDARD,
                starting_flag=flag,
                config=config,
                overwrite=True,
            )

print('\nAll grid points complete.')

Overwrite=True: Deleting existing results file: ATTO-488_qy_0.100_readnoise_0.010000_LM_method_ATTO 488_rawresults.h5
Analysed photon flux 1/1    Time elapsed: 0.162 min                             ?, ?it/s]
Completed analysis of 1 photon flux values    Total time: 0.162 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard


Overwrite=True: Deleting existing results file: ATTO-488_qy_0.100_readnoise_0.011514_LM_method_ATTO 488_rawresults.h5
Analysed photon flux 1/1    Time elapsed: 0.141 min                             ?, ?it/s]
Completed analysis of 1 photon flux values    Total time: 0.141 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard


Overwrite=True: Deleting existing results file: ATTO-488_qy_0.100_readnoise_0.013257_LM_method_ATTO 488_rawresults.h5
Analysed photon flux 1/1    Time elapsed: 0.140 min                             ?, ?it/s]
Completed analysis of 1 photon flux values    Total time: 0.140 min


...

INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard


Analysed photon flux 1/1    Time elapsed: 0.140 min                             ?, ?it/s]
Completed analysis of 1 photon flux values    Total time: 0.140 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard


Analysed photon flux 1/1    Time elapsed: 0.153 min                             ?, ?it/s]
Completed analysis of 1 photon flux values    Total time: 0.153 min


INFO:src.Multicolour_Simulation_Functions:Simulation completed for strategy standard


In [ ]:
# --- Load results and build 2D metric surfaces ---
n_qy = len(peak_qy_space)
n_rn = len(read_noise_space)

# Storage: dict keyed by dye → 2D array (n_qy, n_rn)
sigma_xy_surf    = {}
colour_std_surf  = {}
fit_yield_surf   = {}

for dye in dyes:
    dyestr = dye.replace(' ', '-').replace('/', '-')

    sigma_xy_arr   = np.full((n_qy, n_rn), np.nan)
    colour_std_arr = np.full((n_qy, n_rn), np.nan)
    fit_yield_arr  = np.full((n_qy, n_rn), np.nan)

    for i, peak_qy in enumerate(peak_qy_space):
        for j, rn in enumerate(read_noise_space):
            flag = f'{dyestr}_qy_{peak_qy:.3f}_readnoise_{rn:.6f}_'

            gt_path  = os.path.join(save_folder, f'{flag}LM_method_{dyestr}_fittesting_input_groundtruthpositions.csv')
            raw_path = os.path.join(save_folder, f'{flag}LM_method_{dyestr}_rawresults.h5')
            inp_path = os.path.join(save_folder, f'{flag}LM_method_{dyestr}_fittesting_input_parameters.csv')

            if not (os.path.exists(gt_path) and os.path.exists(raw_path) and os.path.exists(inp_path)):
                continue

            gt      = pd.read_csv(gt_path)
            results = pd.read_hdf(raw_path)
            inp     = pd.read_csv(inp_path).to_numpy()[0]

            results = results[results['photon_level'] == 0]

            x0 = gt['x0'].to_numpy() / pixel_size
            y0 = gt['y0'].to_numpy() / pixel_size

            valid = ~(results['xc'].isna() | results['yc'].isna())
            fit_yield_arr[i, j] = valid.mean()

            filt = valid & (
                (results['xc'] > 0) & (results['xc'] < image_size) &
                (results['yc'] > 0) & (results['yc'] < image_size) &
                (results['s_x'] > 0) & (results['s_y'] > 0)
            )
            if filt.sum() > 10:
                err_x = results['xc'].to_numpy()[filt] - x0[filt]
                err_y = results['yc'].to_numpy()[filt] - y0[filt]
                sigma_xy_arr[i, j] = (
                    np.sqrt((np.nanstd(err_x)**2 + np.nanstd(err_y)**2) / 2) * pixel_size
                )

                dye_BGR = inp[-3:]
                dye_BGR = dye_BGR / np.sum(dye_BGR)
                colour_loc = np.expand_dims(dye_BGR, 0)
                colour = np.vstack([
                    results['A_B'].to_numpy()[filt],
                    results['A_G'].to_numpy()[filt],
                    results['A_R'].to_numpy()[filt],
                ]).T
                colour_std_arr[i, j] = np.nanstd(cdist(colour, colour_loc))

    sigma_xy_surf[dye]   = sigma_xy_arr
    colour_std_surf[dye] = colour_std_arr
    fit_yield_surf[dye]  = fit_yield_arr
    print(f'{dye}: yield {np.nanmin(fit_yield_arr):.3f}–{np.nanmax(fit_yield_arr):.3f}'
          f'  σ_xy {np.nanmin(sigma_xy_arr):.1f}–{np.nanmax(sigma_xy_arr):.1f} nm')

print('Done.')

In [ ]:
# --- Plot 2D surfaces: rows = dyes, columns = fit yield / sigma_xy / sigma_colour ---
import matplotlib.colors as mcolors

metrics = [
    ('fit_yield',    fit_yield_surf,   'Fit yield',        r'Fit yield',          '%',   'viridis',  (0,   1)),
    ('sigma_xy',     sigma_xy_surf,    r'$\sigma_{xy}$',   r'$\sigma_{xy}$ / nm', 'nm',  'plasma',   None),
    ('sigma_colour', colour_std_surf,  r'$\sigma_{colour}$', r'$\sigma_{colour}$', '',   'cividis',  None),
]

fig, axs = plt.subplots(
    len(dyes), len(metrics),
    figsize=(4.5 * len(metrics), 3.2 * len(dyes)),
    constrained_layout=True,
)

# Build pcolormesh grid edges (for log-x axis)
rn_edges = np.concatenate([
    [read_noise_space[0] * (read_noise_space[0] / read_noise_space[1])],
    np.sqrt(read_noise_space[:-1] * read_noise_space[1:]),
    [read_noise_space[-1] * (read_noise_space[-1] / read_noise_space[-2])],
])
qy_step = peak_qy_space[1] - peak_qy_space[0]
qy_edges = np.concatenate([
    [peak_qy_space[0]  - qy_step / 2],
    (peak_qy_space[:-1] + peak_qy_space[1:]) / 2,
    [peak_qy_space[-1] + qy_step / 2],
])

for col, (key, surf_dict, col_title, cbar_label, unit, cmap, clim) in enumerate(metrics):
    for row, dye in enumerate(dyes):
        ax  = axs[row, col]
        data = surf_dict[dye].copy()

        if key == 'fit_yield':
            data = data * 100   # convert to %

        vmin, vmax = (clim if clim is not None else
                      (np.nanpercentile(data, 2), np.nanpercentile(data, 98)))

        pcm = ax.pcolormesh(
            rn_edges, qy_edges, data,
            cmap=cmap, vmin=vmin, vmax=vmax,
            shading='flat',
        )
        ax.set_xscale('log')

        cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
        cbar.set_label(cbar_label + (f' ({unit})' if unit else ''), fontsize=7)
        cbar.ax.tick_params(labelsize=6)

        ax.set_xlabel('Read noise (RMS e⁻)', fontsize=8)
        ax.set_ylabel('Peak pixel QY', fontsize=8)
        ax.tick_params(labelsize=7)

        if row == 0:
            ax.set_title(col_title, fontsize=9, fontweight='bold')
        ax.text(0.03, 0.97, dye, transform=ax.transAxes,
                fontsize=7, va='top', ha='left',
                bbox=dict(fc='white', ec='none', alpha=0.7, pad=1))

folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Papers/Multicolour/SI/Lowest_Noise_Level/'
os.makedirs(folder, exist_ok=True)
plt.savefig(os.path.join(folder, 'Noise_QY_Surface.svg'), dpi=300, format='svg')
plt.savefig(os.path.join(folder, 'Noise_QY_Surface.png'), dpi=200)
plt.show()